# NB4 — `hybrid_crossattn` Model Definition + Sanity Check

**Status:** model definition + dummy-tensor forward-pass shape check ONLY.
**No real training, no Drive mount, no dataset loading in this notebook.**

Per the locked handoff (`GastroNet_FINAL_Handoff.md`, Section 7 / NB4):

- Unidirectional cross-attention: **CNN spatial feature tokens as queries**,
  attending to **ViT-Small token keys/values** (not bidirectional).
- A learned 2D positional embedding is added to the flattened CNN spatial
  tokens before the attention block.
- Fusion block kept small given dataset size: **1–2 attention layers**,
  `d_model=256` (ViT's 384 is projected down to 256, not the reverse).
- **Content-only queries** — CNN tokens are not also projected into ViT's
  positional space. Simpler; revisit only if results are weak.
- Reuses the `GastroDualDataset` pattern from NB3 (dual native resolution:
  448×448 for the CNN branch, 224×224 for the ViT branch) — no new dataset
  class needed. The class is reproduced here (unchanged from NB3) purely so
  this notebook is self-contained for the shape check; NB5 should import the
  real one from wherever NB3 defines/saves it rather than redefining it.

Architectural decisions this notebook does **not** relitigate:
- ViT-Small (not ViT-Base) via `timm`.
- Dual native resolution (448 CNN / 224 ViT), never forced to a single size.


In [1]:
# One-time per session/notebook, per Decision A in the handoff.
!pip install timm --break-system-packages -q


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchvision

print("torch:", torch.__version__)
print("timm:", timm.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


torch: 2.11.0+cu128
timm: 1.0.28
CUDA available: True
Using device: cuda


## 1. `GastroDualDataset` (reused, unchanged from NB3)

Returns `(cnn_tensor_448, vit_tensor_224, label)` per sample. Both resized
copies of a source image share one set of augmentation decisions (flip,
rotation, brightness/contrast), so the two branches see consistent augmented
views rather than two independently randomized ones.

This notebook does not touch real data — the class is included only so the
dummy-tensor check below exercises the exact tensor shapes/dtypes this
dataset actually produces. NB5 should import this from the shared location
NB3 saved it to, not redefine it.


In [3]:
from PIL import Image
import random
import torchvision.transforms.functional as TF

class GastroDualDataset(torch.utils.data.Dataset):
    """
    Dual native-resolution dataset (Decision B).

    Returns (cnn_tensor, vit_tensor, label):
      - cnn_tensor: 3x448x448, normalized for EfficientNet-B4
      - vit_tensor: 3x224x224, normalized for ViT-Small (timm defaults)
    One shared set of augmentation decisions is applied identically to both
    resized copies of the same source image.
    """

    CNN_SIZE = 448
    VIT_SIZE = 224

    # ImageNet normalization stats -- correct for both EfficientNet-B4 and
    # timm's vit_small_patch16_224 pretrained weights.
    MEAN = [0.485, 0.456, 0.406]
    STD = [0.229, 0.224, 0.225]

    def __init__(self, samples, train=True):
        """
        samples: list of (path, label) tuples, already remapped to the local
                 on-disk copy per the infra rules (never read from Drive
                 directly during training).
        train: if True, apply the shared random augmentation; if False
               (val/test), deterministic resize + normalize only.
        """
        self.samples = samples
        self.train = train

    def __len__(self):
        return len(self.samples)

    def _augment_params(self):
        return {
            "hflip": self.train and random.random() < 0.5,
            "angle": random.uniform(-15, 15) if self.train else 0.0,
            "brightness": random.uniform(0.85, 1.15) if self.train else 1.0,
            "contrast": random.uniform(0.85, 1.15) if self.train else 1.0,
        }

    def _apply(self, img, size, params):
        img = img.resize((size, size), Image.BILINEAR)
        if params["hflip"]:
            img = TF.hflip(img)
        if params["angle"] != 0.0:
            img = TF.rotate(img, params["angle"])
        if self.train:
            img = TF.adjust_brightness(img, params["brightness"])
            img = TF.adjust_contrast(img, params["contrast"])
        tensor = TF.to_tensor(img)
        tensor = TF.normalize(tensor, self.MEAN, self.STD)
        return tensor

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        params = self._augment_params()  # one shared decision, both branches
        cnn_tensor = self._apply(img, self.CNN_SIZE, params)
        vit_tensor = self._apply(img, self.VIT_SIZE, params)
        return cnn_tensor, vit_tensor, label


## 2. CNN branch — EfficientNet-B4 spatial feature map (queries source)

Unlike NB3's `hybrid_concat` (which pools the CNN branch straight to a
1792-dim vector), `hybrid_crossattn` needs the CNN's **spatial** feature map
*before* pooling, so each spatial location can act as a query token.

For `torchvision`'s EfficientNet-B4 at 448x448 input, `features` outputs a
`(B, 1792, 14, 14)` map (448 / 32 stride = 14). We verify this shape
directly below rather than hard-coding it blind.


In [4]:
class CNNSpatialEncoder(nn.Module):
    """EfficientNet-B4 backbone, returns the pre-pool spatial feature map."""

    def __init__(self, pretrained=True):
        super().__init__()
        weights = torchvision.models.EfficientNet_B4_Weights.DEFAULT if pretrained else None
        backbone = torchvision.models.efficientnet_b4(weights=weights)
        self.features = backbone.features  # (B, 1792, H, W) at 448 input
        self.out_channels = 1792

    def forward(self, x):
        return self.features(x)  # (B, 1792, H, W)


# Verify the spatial shape at 448x448 before building anything on top of it.
_cnn_probe = CNNSpatialEncoder(pretrained=False).to(device).eval()
with torch.no_grad():
    _probe_out = _cnn_probe(torch.randn(2, 3, 448, 448, device=device))
print("CNN spatial feature map shape:", tuple(_probe_out.shape))
CNN_GRID_H, CNN_GRID_W = _probe_out.shape[-2], _probe_out.shape[-1]
CNN_NUM_TOKENS = CNN_GRID_H * CNN_GRID_W
print(f"-> CNN grid: {CNN_GRID_H}x{CNN_GRID_W} = {CNN_NUM_TOKENS} spatial tokens")
del _cnn_probe, _probe_out


CNN spatial feature map shape: (2, 1792, 14, 14)
-> CNN grid: 14x14 = 196 spatial tokens


## 3. ViT branch — ViT-Small patch tokens (keys/values source)

`timm`'s `vit_small_patch16_224` with `num_classes=0` returns a pooled
384-dim vector by default (used as-is in NB3's `hybrid_concat`). Here we
instead need the **full patch token sequence** to serve as keys/values, so
we call `forward_features` and drop the CLS token, keeping only the 196
patch tokens (`224/16 = 14` -> `14x14 = 196`).


In [5]:
class ViTTokenEncoder(nn.Module):
    """ViT-Small (timm) backbone, returns patch tokens with CLS dropped."""

    def __init__(self, pretrained=True):
        super().__init__()
        self.vit = timm.create_model(
            "vit_small_patch16_224", pretrained=pretrained, num_classes=0
        )
        self.embed_dim = self.vit.embed_dim  # 384
        self.num_prefix_tokens = getattr(self.vit, "num_prefix_tokens", 1)  # CLS (+ maybe dist)

    def forward(self, x):
        tokens = self.vit.forward_features(x)  # (B, 1 + N, 384) typically
        patch_tokens = tokens[:, self.num_prefix_tokens:, :]  # drop CLS (and dist, if any)
        return patch_tokens  # (B, N, 384)


_vit_probe = ViTTokenEncoder(pretrained=False).to(device).eval()
with torch.no_grad():
    _vit_out = _vit_probe(torch.randn(2, 3, 224, 224, device=device))
print("ViT patch token sequence shape:", tuple(_vit_out.shape))
VIT_NUM_TOKENS = _vit_out.shape[1]
VIT_EMBED_DIM = _vit_out.shape[2]
print(f"-> ViT: {VIT_NUM_TOKENS} patch tokens, embed_dim={VIT_EMBED_DIM}")
del _vit_probe, _vit_out


ViT patch token sequence shape: (2, 196, 384)
-> ViT: 196 patch tokens, embed_dim=384


## 4. Learned 2D positional embedding for CNN tokens

Per the design: a learned 2D positional embedding is added to the flattened
CNN spatial tokens *before* the attention block. We learn separate row/column
embeddings and sum them per grid cell (standard, parameter-cheap 2D
positional embedding), rather than one embedding per flattened position —
cheaper and generalizes slightly better on a small (~4000 image) dataset.


In [6]:
class Learned2DPositionalEmbedding(nn.Module):
    """Separable row/column learned positional embedding for a HxW grid."""

    def __init__(self, grid_h, grid_w, dim):
        super().__init__()
        self.grid_h = grid_h
        self.grid_w = grid_w
        self.row_embed = nn.Parameter(torch.zeros(grid_h, dim))
        self.col_embed = nn.Parameter(torch.zeros(grid_w, dim))
        nn.init.trunc_normal_(self.row_embed, std=0.02)
        nn.init.trunc_normal_(self.col_embed, std=0.02)

    def forward(self):
        # (H, 1, D) + (1, W, D) -> (H, W, D) -> (H*W, D)
        pos = self.row_embed[:, None, :] + self.col_embed[None, :, :]
        return pos.reshape(self.grid_h * self.grid_w, -1)  # (H*W, D)


## 5. Unidirectional cross-attention block

CNN tokens (projected to `d_model=256`) are the **queries**. ViT tokens
(projected 384->256) are the **keys/values**. Content-only queries: the CNN
tokens are *not* additionally projected into ViT's positional space -- kept
deliberately simple per the design note, to revisit only if results are
weak. Positional information for the queries comes solely from the learned
2D positional embedding added to the CNN tokens beforehand (Section 4);
`nn.MultiheadAttention` itself is positionally agnostic.

1-2 of these blocks are stacked (configurable, default 2), each followed by
a residual + LayerNorm and a small feed-forward block -- a standard
post-norm transformer decoder-style layer, but attention-only in one
direction (no self-attention among CNN tokens, to keep the block small given
dataset size).


In [7]:
class CrossAttentionBlock(nn.Module):
    """
    One unidirectional cross-attention layer:
      query = CNN tokens (d_model), key/value = ViT tokens (projected to d_model)
    followed by residual+LayerNorm and a feed-forward sublayer (also residual+LN).
    """

    def __init__(self, d_model=256, num_heads=4, ff_mult=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * ff_mult),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * ff_mult, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, cnn_tokens, vit_tokens):
        # cnn_tokens: (B, N_cnn, d_model) queries
        # vit_tokens: (B, N_vit, d_model) keys/values
        attn_out, attn_weights = self.attn(
            query=cnn_tokens, key=vit_tokens, value=vit_tokens, need_weights=True
        )
        x = self.norm1(cnn_tokens + self.dropout(attn_out))
        x = self.norm2(x + self.ff(x))
        return x, attn_weights


## 6. Full `hybrid_crossattn` model

Pipeline:
1. CNN branch -> spatial feature map -> flatten to tokens -> project to
   `d_model` -> add learned 2D positional embedding.
2. ViT branch -> patch tokens -> project 384->`d_model` (content-only, no
   positional projection).
3. Stack of `num_layers` (1-2) `CrossAttentionBlock`s, CNN tokens attending
   to ViT tokens, updated each layer.
4. Mean-pool the final attended CNN tokens -> classifier head
   (`Linear(d_model, 4)`), consistent in spirit with NB3's small MLP head
   but sized for `d_model=256` since pooling happens after fusion here
   rather than concatenating two separately-pooled vectors.

`num_layers` defaults to 2 per the "1-2 attention layers" guidance; kept as
a constructor argument so NB5 can ablate 1 vs 2 layers cheaply if desired.


In [8]:
class HybridCrossAttnModel(nn.Module):
    def __init__(
        self,
        num_classes=4,
        d_model=256,
        num_heads=4,
        num_layers=2,
        cnn_grid=(CNN_GRID_H, CNN_GRID_W),
        dropout=0.1,
        pretrained_backbones=True,
    ):
        super().__init__()
        self.cnn_encoder = CNNSpatialEncoder(pretrained=pretrained_backbones)
        self.vit_encoder = ViTTokenEncoder(pretrained=pretrained_backbones)

        cnn_channels = self.cnn_encoder.out_channels  # 1792
        vit_dim = self.vit_encoder.embed_dim  # 384
        grid_h, grid_w = cnn_grid

        # Project CNN spatial tokens -> d_model (content-only queries: no
        # separate positional projection branch).
        self.cnn_proj = nn.Linear(cnn_channels, d_model)
        self.pos_embed = Learned2DPositionalEmbedding(grid_h, grid_w, d_model)

        # Project ViT tokens (384) -> d_model (256), per the design note:
        # project ViT's dim down to d_model, not the reverse.
        self.vit_proj = nn.Linear(vit_dim, d_model)

        self.blocks = nn.ModuleList(
            [
                CrossAttentionBlock(d_model=d_model, num_heads=num_heads, dropout=dropout)
                for _ in range(num_layers)
            ]
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )

        self.d_model = d_model
        self.grid_h, self.grid_w = grid_h, grid_w

    def forward(self, cnn_input, vit_input, return_attn=False):
        # --- CNN branch: spatial map -> tokens ---
        cnn_feat_map = self.cnn_encoder(cnn_input)          # (B, C, H, W)
        B, C, H, W = cnn_feat_map.shape
        cnn_tokens = cnn_feat_map.flatten(2).transpose(1, 2)  # (B, H*W, C)
        cnn_tokens = self.cnn_proj(cnn_tokens)                # (B, H*W, d_model)
        cnn_tokens = cnn_tokens + self.pos_embed()[None, :, :]  # + learned 2D pos emb

        # --- ViT branch: patch tokens -> projected to d_model ---
        vit_tokens = self.vit_encoder(vit_input)   # (B, N_vit, 384)
        vit_tokens = self.vit_proj(vit_tokens)     # (B, N_vit, d_model)

        # --- Cross-attention stack: CNN tokens (queries) attend to ViT (kv) ---
        attn_maps = []
        x = cnn_tokens
        for block in self.blocks:
            x, attn_weights = block(x, vit_tokens)
            attn_maps.append(attn_weights)

        # --- Pool + classify ---
        pooled = x.mean(dim=1)          # (B, d_model)
        logits = self.classifier(pooled)  # (B, num_classes)

        if return_attn:
            return logits, attn_maps
        return logits


## 7. Dummy-tensor sanity check

NB4's entire job, per the handoff: verify every piece (positional embedding,
cross-attention block, dimension projections, full model) produces correct
shapes on fake data **before spending any GPU time on real training**. No
real dataset is touched here.


In [9]:
BATCH_SIZE_PROBE = 4
NUM_CLASSES = 4

model = HybridCrossAttnModel(
    num_classes=NUM_CLASSES,
    d_model=256,
    num_heads=4,
    num_layers=2,
    cnn_grid=(CNN_GRID_H, CNN_GRID_W),
    pretrained_backbones=False,  # False here: shape check only, no need to download weights
).to(device)
model.eval()

dummy_cnn = torch.randn(BATCH_SIZE_PROBE, 3, 448, 448, device=device)
dummy_vit = torch.randn(BATCH_SIZE_PROBE, 3, 224, 224, device=device)

with torch.no_grad():
    logits, attn_maps = model(dummy_cnn, dummy_vit, return_attn=True)

print("Input  cnn_tensor:", tuple(dummy_cnn.shape))
print("Input  vit_tensor:", tuple(dummy_vit.shape))
print("Output logits:    ", tuple(logits.shape), "-> expected", (BATCH_SIZE_PROBE, NUM_CLASSES))
print()
for i, w in enumerate(attn_maps):
    print(f"Cross-attn layer {i} weights shape:", tuple(w.shape),
          f"(expected ({BATCH_SIZE_PROBE}, {CNN_NUM_TOKENS}, {VIT_NUM_TOKENS}))")

assert logits.shape == (BATCH_SIZE_PROBE, NUM_CLASSES), "Logits shape mismatch!"
for w in attn_maps:
    assert w.shape == (BATCH_SIZE_PROBE, CNN_NUM_TOKENS, VIT_NUM_TOKENS), "Attention map shape mismatch!"

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print()
print(f"Total params:     {n_params:,}")
print(f"Trainable params: {n_trainable:,}")
print()
print("ALL SHAPE CHECKS PASSED.")


Input  cnn_tensor: (4, 3, 448, 448)
Input  vit_tensor: (4, 3, 224, 224)
Output logits:     (4, 4) -> expected (4, 4)

Cross-attn layer 0 weights shape: (4, 196, 196) (expected (4, 196, 196))
Cross-attn layer 1 weights shape: (4, 196, 196) (expected (4, 196, 196))

Total params:     41,359,564
Trainable params: 41,359,564

ALL SHAPE CHECKS PASSED.


In [10]:
# Sanity check: gradient flows end-to-end through both branches and the
# fusion block (catches any accidental detach / no_grad leftover).
model.train()
dummy_labels = torch.randint(0, NUM_CLASSES, (BATCH_SIZE_PROBE,), device=device)
criterion = nn.CrossEntropyLoss()

logits = model(dummy_cnn, dummy_vit)
loss = criterion(logits, dummy_labels)
loss.backward()

n_with_grad = sum(1 for p in model.parameters() if p.requires_grad and p.grad is not None)
n_total_trainable = sum(1 for p in model.parameters() if p.requires_grad)
print(f"Params with a populated .grad after backward(): {n_with_grad} / {n_total_trainable}")
assert n_with_grad == n_total_trainable, "Some trainable params did not receive gradients!"
print(f"Dummy loss: {loss.item():.4f}")
print("BACKWARD PASS OK -- gradients reach every trainable parameter.")

model.zero_grad()


Params with a populated .grad after backward(): 598 / 598
Dummy loss: 1.2953
BACKWARD PASS OK -- gradients reach every trainable parameter.


## 8. Loading pretrained backbones (verification only, still no training)

Quick check that the real pretrained-weight path (as will actually be used
in NB5) also builds and runs cleanly, since the shape checks above used
`pretrained_backbones=False` to avoid an unnecessary weights download during
iteration. Run this cell only if you want to confirm the pretrained path
before moving on to NB5 -- it downloads both backbones' pretrained weights.


In [11]:
model_pretrained = HybridCrossAttnModel(
    num_classes=NUM_CLASSES,
    d_model=256,
    num_heads=4,
    num_layers=2,
    cnn_grid=(CNN_GRID_H, CNN_GRID_W),
    pretrained_backbones=True,
).to(device)
model_pretrained.eval()

with torch.no_grad():
    logits_pt = model_pretrained(dummy_cnn, dummy_vit)

print("Pretrained-backbone logits shape:", tuple(logits_pt.shape))
assert logits_pt.shape == (BATCH_SIZE_PROBE, NUM_CLASSES)
print("Pretrained backbone path OK.")


Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 166MB/s]


model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Pretrained-backbone logits shape: (4, 4)
Pretrained backbone path OK.


## 9. Next steps

- **NB4 is done once every check above passes.** No training happens here.
- NB5 builds on this exact model definition:
  `MODEL_FAMILY = "hybrid_crossattn_v2"`, `SEEDS_TO_RUN = [42, 123, 7]`,
  reusing the standard training-notebook pattern (Section 11 of the
  handoff): mount Drive, local dataset copy, `dataset_split_v2.json`,
  `GastroDualDataset`, `resume_or_start`/`finalize_experiment` via
  `checkpoint_utils`, `PATIENCE=6`, per-batch progress prints,
  `assert_split_hash_matches` before eval, and raw `predictions`/`labels`
  saved into `results.json` for the later significance testing in NB6.
- Decide whether additional seeds beyond the standard 3 are worth it only
  after seeing whether `hybrid_crossattn_v2` meaningfully differs from
  `hybrid_concat_v2` -- per Section 3, a non-improvement is a valid,
  reportable finding, not a reason to keep tweaking.
